# 02 · Validation against Copernicus EMS

How good is the severity map of notebook 01? Two independent references, each measuring a different face of the error:

| | Reference | Image | What it measures | Where |
|---|---|---|---|---|
| **M1** | EMSN188, building damage points (P08) | Pléiades Neo, 0.3 m | omission in the urban interface | "Las Tablas" area of interest only |
| **M2** | EMSR715, burned area delineation v2 (AOI01) | WorldView-2, 2 m, 5 Feb 2024 | burned / unburned agreement, commission and omission | the four municipalities, inside the WorldView-2 footprint |

EMSR715 used a Sentinel-2 scene as its *pre*-event reference, so its independence from this map is partial, not total; its post-event image is not one of ours.

**The protocol was written down before any result was seen**, including the rule that picks the headline figure. Nothing below was tuned after looking at the numbers.

The references are **not redistributed** with this repository: you download them from Copernicus and the notebook checks that they are the same files that produced the published metrics.

In [1]:
import json, logging, os, warnings
from pathlib import Path
logging.basicConfig(level=logging.WARNING, format="%(message)s")
# rasterio 1.5 triggers a NumPy 2.5 deprecation notice that does not affect the results
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import rasterio
from rasterio.features import rasterize
from shapely.ops import unary_union
from severidad import calculo, estudio, fuente, validacion   # installed with `pip install -e .`

RAW = Path(os.environ.get("INCENDIO_DATOS", "../datos_crudos")).resolve()
REFS = RAW / "referencias"
REFS.mkdir(parents=True, exist_ok=True)
PUBLISHED = json.loads(Path("../datos/metricas.json").read_text(encoding="utf-8"))
P = PUBLISHED["protocol"]
SEED = P["seed"]
N = estudio.SAMPLE_SIZE
THRESHOLDS = dict(estudio.THRESHOLDS)
CUT = dict(estudio.MUNICIPALITIES)

## 1. Download the references (by hand, once)

Both are free and need no account. Save each file **with the name shown** in `datos_crudos/referencias/` (or in `$INCENDIO_DATOS/referencias/`), without unzipping it.

| File | Where | What to click |
|---|---|---|
| `EMSR715_products.zip` (178 MB) | [EMSR715, Wildfire in Valparaiso region, Chile](https://mapping.emergency.copernicus.eu/activations/EMSR715) | "All products (ZIP)" |
| `EMSN188_GeospatialData.zip` (1.3 MB) | [EMSN188, Post-wildfire damage assessment in Valparaiso region, Chile](https://mapping.emergency.copernicus.eu/activations/EMSN188) | "Geospatial data" |

**Terms of use.** Copernicus EMS On-Demand Mapping data can be reproduced, distributed, adapted and combined with other data free of charge; whoever communicates it must tell the recipients its source. The data is provided "as is", without warranty ([CEMS terms and conditions](https://mapping.emergency.copernicus.eu/terms-and-conditions/), [citation guidelines](https://mapping.emergency.copernicus.eu/about/citation-guidelines/)). This notebook uses:

- Copernicus Emergency Management Service (© 2024 European Union), [EMSR715] Valparaiso (AOI01): Delineation Map, version 2.
- Copernicus Emergency Management Service (© 2024 European Union), [EMSN188] Post-wildfire damage assessment in Valparaiso region, Chile: P08 Detailed damage assessment.

If Copernicus repackages the EMSR715 download, the outer ZIP can change while the product inside does not; the check below then falls back to the delineation product itself.

In [2]:
r715, r188 = PUBLISHED["references"]["EMSR715"], PUBLISHED["references"]["EMSN188"]
EMSR715, EMSN188 = REFS / r715["file"], REFS / r188["file"]

validacion.check_file(EMSN188, r188["bytes"], r188["sha256"])
print(f"{EMSN188.name}: same file as the published metrics")
try:
    validacion.check_file(EMSR715, r715["bytes"], r715["sha256"])
    print(f"{EMSR715.name}: same file as the published metrics")
except ValueError as outer_differs:
    if validacion.inner_sha256(EMSR715, r715["product"]) != r715["product_sha256"]:
        raise ValueError(f"{r715['product']} inside {EMSR715.name} is not the one used: {outer_differs}")
    print(f"{EMSR715.name} was repackaged, but {r715['product']} inside it is the one used")

EMSN188_GeospatialData.zip: same file as the published metrics


EMSR715_products.zip: same file as the published metrics


## 2. The map being validated and the study area

The published classes (`datos/clases.tif`) are used, after checking their content hash: it is the same raster that notebook 01 rebuilds from scratch.

In [3]:
classes_path = Path("../datos/clases.tif")
assert calculo.content_hash(classes_path) == P["classes_content_sha256"], "datos/clases.tif is not the published map"
with rasterio.open(classes_path) as r:
    classes, T, CRS_MAP = r.read(1), r.transform, r.crs.to_string()
HA = abs(T.a * T.e) / 10_000.0

local = RAW / "comunas.geojson"
if not local.exists():
    local.write_text(json.dumps(fuente.municipalities(CUT)), encoding="utf-8")
municipalities = json.loads(local.read_text(encoding="utf-8"))
assert {f["properties"]["CUT_COM"] for f in municipalities["features"]} == set(P["municipalities"])
in_area = validacion.zone_mask(municipalities, CRS_MAP, T, classes.shape)
print(f"grid {classes.shape[1]} x {classes.shape[0]} px of {abs(T.a):.0f} m in {CRS_MAP}; "
      f"study area {in_area.sum() * HA:,.1f} ha")

grid 1756 x 1261 px of 20 m in EPSG:32719; study area 70,313.4 ha

## 3. M1 · Omission in the urban interface (EMSN188 building damage)

Universe: every damage point inside the four municipalities whose pixel has valid data (points on "no data" are counted and reported apart). Sample: 300 points drawn with the seed, keeping a point only if it is 100 m or more from every point already kept. Result: share of the sample that falls in a burned class, for each threshold, with a Wilson 95 % interval; and the same for "Destroyed" buildings only.

Damage grades as stated in the EMSN188 technical report: 3 Destroyed, 2 Damaged, 1 Possible damage.

In [4]:
xy, grades = validacion.emsn188_damage(EMSN188, CRS_MAP)
cls_at, on_grid = validacion.values_at(xy, classes, T)
zone_at, _ = validacion.values_at(xy, in_area.astype(np.uint8), T)
inside = on_grid & (zone_at > 0)
no_data = inside & (cls_at == 0)
universe = np.flatnonzero(inside & (cls_at > 0))
sample = universe[validacion.sample_points(xy[universe], N, estudio.M1_MIN_DISTANCE_M, SEED)]
destroyed = sample[grades[sample] == "3"]

m1 = {"points": len(xy), "in_municipalities": int(inside.sum()), "in_no_data": int(no_data.sum()),
      "universe": len(universe), "sample": len(sample),
      "sample_grades": {name: int((grades[sample] == code).sum()) for code, name in validacion.DAMAGE_GRADES.items()}}
print(f"{m1['points']} damage points, {m1['in_municipalities']} in the municipalities, "
      f"{m1['in_no_data']} on no data; sample of {m1['sample']}: {m1['sample_grades']}")
for t, code in THRESHOLDS.items():
    p, d = validacion.wilson(int((cls_at[sample] >= code).sum()), len(sample)), validacion.wilson(int((cls_at[destroyed] >= code).sum()), len(destroyed))
    m1[t] = {"in_burned_class": p.successes, "n": p.n, "pct": round(100 * p.value, 1), "wilson": [round(100 * p.low, 1), round(100 * p.high, 1)],
             "destroyed_in_burned_class": d.successes, "destroyed_n": d.n, "destroyed_pct": round(100 * d.value, 1),
             "destroyed_wilson": [round(100 * d.low, 1), round(100 * d.high, 1)]}
    print(f"threshold {t} ({P['thresholds'][t]}): damage points in a burned class {p}; Destroyed only {d}")

7275 damage points, 7275 in the municipalities, 5 on no data; sample of 300: {'Destroyed': 203, 'Damaged': 30, 'Possible damage': 67}
threshold A (all burned classes (code >= 4)): damage points in a burned class 164/300 = 54.7 % (Wilson 95 %: 49.0 to 60.2); Destroyed only 123/203 = 60.6 % (Wilson 95 %: 53.7 to 67.1)
threshold B (moderate-low severity or higher (code >= 5)): damage points in a burned class 42/300 = 14.0 % (Wilson 95 %: 10.5 to 18.4); Destroyed only 32/203 = 15.8 % (Wilson 95 %: 11.4 to 21.4)


Even with every burned class, close to half of the damaged buildings fall outside the burned area. This is expected: NBR responds to the loss of vegetation, and an urban pixel (roofs, streets, little vegetation before the fire) changes little even when the houses burn. **This map measures severity on vegetation; building damage needs very high resolution imagery** (limitation O1).

## 4. M2 · Burned / unburned against the EMSR715 delineation

- Domain: valid pixels inside the four municipalities and inside the WorldView-2 footprint.
- Reference: a pixel is burned if its centre falls in an `observedEventA` polygon of the delineation v2 (which already excludes the area burned before the event).
- Sample: 300 pixels per predicted stratum (burned, unburned), one generator with the seed, burned stratum first, minimum Chebyshev distance of 10 px (200 m).
- Metrics: overall accuracy of the sample and commission with Wilson 95 %; area-weighted accuracy and omission (Olofsson et al. 2014), which estimate those of the whole map.

In [5]:
burned_poly, footprint = validacion.emsr715_delineation(EMSR715, CRS_MAP)
reference = rasterize([(g, 1) for g in burned_poly], out_shape=classes.shape, transform=T, fill=0, dtype="uint8").astype(bool)
in_footprint = rasterize([(unary_union(footprint), 1)], out_shape=classes.shape, transform=T, fill=0, dtype="uint8").astype(bool)
domain = (classes > 0) & in_area & in_footprint
m2 = {"domain_pixels": int(domain.sum()), "domain_ha": round(domain.sum() * HA, 1),
      "reference_burned_ha": round((domain & reference).sum() * HA, 1)}
print(f"domain {m2['domain_ha']:,.1f} ha; reference burned area in the domain {m2['reference_burned_ha']:,.1f} ha")
for t, code in THRESHOLDS.items():
    predicted = classes >= code
    a = validacion.agreement(predicted, reference, domain, N, SEED, estudio.M2_MIN_DISTANCE_PX)
    m2[t] = {"predicted_burned_ha": round((domain & predicted).sum() * HA, 1),
             "overall": [a.overall.successes, a.overall.n], "overall_pct": round(100 * a.overall.value, 1),
             "overall_wilson": [round(100 * a.overall.low, 1), round(100 * a.overall.high, 1)],
             "weighted_accuracy_pct": round(100 * a.weighted_accuracy, 1),
             "commission": [a.commission.successes, a.commission.n], "commission_pct": round(100 * a.commission.value, 1),
             "commission_wilson": [round(100 * a.commission.low, 1), round(100 * a.commission.high, 1)],
             "weighted_omission_pct": round(100 * a.weighted_omission, 1), "matrix": [list(row) for row in a.matrix]}
    print(f"\nthreshold {t} ({P['thresholds'][t]}): predicted burned {m2[t]['predicted_burned_ha']:,.1f} ha")
    print(f"  overall accuracy (sample) {a.overall}")
    print(f"  area-weighted accuracy    {m2[t]['weighted_accuracy_pct']} %")
    print(f"  commission                {a.commission}")
    print(f"  area-weighted omission    {m2[t]['weighted_omission_pct']} %")
    print(f"  matrix (rows predicted burned, unburned; columns reference burned, unburned): {a.matrix}")

domain 69,926.0 ha; reference burned area in the domain 8,600.5 ha

threshold A (all burned classes (code >= 4)): predicted burned 10,663.0 ha
  overall accuracy (sample) 520/600 = 86.7 % (Wilson 95 %: 83.7 to 89.2)
  area-weighted accuracy    95.2 %
  commission                77/300 = 25.7 % (Wilson 95 %: 21.1 to 30.9)
  area-weighted omission    7.0 %
  matrix (rows predicted burned, unburned; columns reference burned, unburned): ((223, 77), (3, 297))



threshold B (moderate-low severity or higher (code >= 5)): predicted burned 6,759.7 ha
  overall accuracy (sample) 558/600 = 93.0 % (Wilson 95 %: 90.7 to 94.8)
  area-weighted accuracy    96.8 %
  commission                35/300 = 11.7 % (Wilson 95 %: 8.5 to 15.8)
  area-weighted omission    19.8 %
  matrix (rows predicted burned, unburned; columns reference burned, unburned): ((265, 35), (7, 293))


## 5. Headline figure (rule fixed before measuring)

The threshold with the higher area-weighted accuracy against EMSR715 is the headline; the other is published as the upper end of the range. A difference under 1 point would have selected A (the complete Key and Benson scheme).

In [6]:
a, b = m2["A"]["weighted_accuracy_pct"], m2["B"]["weighted_accuracy_pct"]
chosen = "A" if abs(a - b) < 1.0 else ("A" if a > b else "B")
other = "A" if chosen == "B" else "B"
headline = {"threshold": chosen, "hectares": m2[chosen]["predicted_burned_ha"], "range_up_to_hectares": max(m2["A"]["predicted_burned_ha"], m2["B"]["predicted_burned_ha"])}
print(f"weighted accuracy A {a} %, B {b} % -> headline threshold {chosen} ({P['thresholds'][chosen]})")
print(f"Headline: {headline['hectares']:,.1f} ha, range up to {headline['range_up_to_hectares']:,.1f} ha; "
      f"EMSR715 reference in the same domain: {m2['reference_burned_ha']:,.1f} ha")

weighted accuracy A 95.2 %, B 96.8 % -> headline threshold B (moderate-low severity or higher (code >= 5))
Headline: 6,759.7 ha, range up to 10,663.0 ha; EMSR715 reference in the same domain: 8,600.5 ha


## 6. Reproducibility check against the published metrics

In [7]:
computed = {"measurement_1": m1, "measurement_2": m2, "headline": headline}
differences = [k for k in computed if computed[k] != PUBLISHED[k]]
print("IDENTICAL to the published metrics" if not differences else f"DIFFERENT from the published metrics: {differences}")

IDENTICAL to the published metrics


## 7. What the validation says

- **O1. Building damage is out of reach.** With every burned class, only about 55 % of the EMSN188 damage points fall in the burned area (49.0 to 60.2 %). A 20 m vegetation index does not see houses burning.
- **O2. The low-severity class mostly adds commission.** Including it raises commission from 11.7 % to 25.7 %; excluding it raises omission from 7.0 % to 19.8 %. No single threshold reproduces the reference: B (6,760 ha) underestimates it and A (10,663 ha) overestimates it, and the EMSR715 figure (8,600 ha in the same domain) lies inside the range.
- **O3.** The Key and Benson class limits were defined for North American ecosystems and are not calibrated for central Chile; the range above is the honest way to report that.
- Spatial scope: M1 covers only the EMSN188 area of interest ("Las Tablas", Viña del Mar), and M2 only the part of the study area inside the WorldView-2 footprint (all of Viña del Mar and Villa Alemana, 88 % of Valparaíso, 69 % of Quilpué).

Contains modified Copernicus Sentinel data 2024. Validation references: Copernicus Emergency Management Service (© 2024 European Union), EMSR715 and EMSN188.